# FL Client 2 - Federated Learning Client

Notebook ini menjalankan **FL Client 2** yang bertugas:
1. Download **Global Model** dari server
2. Melakukan **Local Training** pada data lokal (data/client_2)
3. Upload **weights** ke server untuk aggregation
4. Repeat untuk beberapa rounds

## PENTING (Google Colab):
1. Jalankan `server.ipynb` **TERLEBIH DAHULU**!
2. **COPY URL ngrok** yang muncul di server notebook
3. **PASTE URL** di cell Configuration di bawah
4. Jalankan semua cell

---

## 1. Install Dependencies

In [1]:
# Install dependencies
!pip install efficientnet_pytorch requests -q

  Preparing metadata (setup.py) ... done


## 2. Download Dataset dari GitHub

In [2]:
# ========================================
# DOWNLOAD DATASET DARI GITHUB (SPARSE CHECKOUT)
# ========================================
# Client 2 hanya membutuhkan data client_2

import os

# URL GitHub repository (GANTI DENGAN URL REPO ANDA)
GITHUB_REPO_URL = 'https://github.com/nashuhainsani/test'
DATA_FOLDER = 'day-3/data/client_2'  # Data untuk Client 2
LOCAL_DIR = 'workshop-data'

# Download hanya folder data menggunakan sparse checkout
if not os.path.exists(LOCAL_DIR):
    print('Downloading Client 2 dataset...')
    !git clone --filter=blob:none --sparse {GITHUB_REPO_URL} {LOCAL_DIR}
    %cd {LOCAL_DIR}
    !git sparse-checkout set {DATA_FOLDER}
    %cd ..
    print('Download selesai!')
else:
    print(f'Data sudah ada di folder {LOCAL_DIR}/')

# Set BASE_DIR ke folder yang berisi data
BASE_DIR = os.path.join(LOCAL_DIR, 'day-3')
print(f'Data directory: {BASE_DIR}/')
print(f'Client 2 data: {os.path.join(BASE_DIR, "data/client_2")}')

Cloning into 'workshop-data'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 23 (delta 2), reused 7 (delta 2), pack-reused 13 (from 1)
Receiving objects: 100% (23/23), 39.49 KiB | 19.74 MiB/s, done.
Resolving deltas: 100% (2/2), done.
remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 2 (delta 0), reused 0 (delta 0), pack-reused 1 (from 1)
Receiving objects: 100% (2/2), 250 bytes | 125.00 KiB/s, done.
/content/workshop-data
remote: Enumerating objects: 701, done.
remote: Total 701 (delta 0), reused 0 (delta 0), pack-reused 701 (from 1)
Receiving objects: 100% (701/701), 277.45 MiB | 21.07 MiB/s, done.
Updating files: 100% (703/703), done.
/content
Download selesai!
Data directory: workshop-data/day-3/
Client 2 data: workshop-data/day-3/data/client_2


## 3. Import Libraries

In [3]:
import os
import io
import gzip
import base64
import hashlib
import random
import requests
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from efficientnet_pytorch import EfficientNet
import time
print('Libraries imported!')

Libraries imported!


## 4. Configuration (PASTE SERVER URL DI SINI!)

In [4]:
# ========================================
# PENTING: PASTE SERVER URL DARI NOTEBOOK SERVER!
# ========================================
# Jalankan server.ipynb dulu, lalu copy URL ngrok-nya ke sini

SERVER_URL = 'https://fidelity-scoured-moonbeam.ngrok-free.dev'  # Contoh: 'https://xxxx-xx-xx-xxx-xx.ngrok-free.app'

# Validasi URL
if 'PASTE' in SERVER_URL or 'ngrok' not in SERVER_URL:
    print('='*60)
    print('WARNING: SERVER_URL belum diisi!')
    print('1. Jalankan server.ipynb terlebih dahulu')
    print('2. Copy URL ngrok yang muncul')
    print('3. Paste di variabel SERVER_URL di atas')
    print('='*60)
else:
    print(f'Server URL: {SERVER_URL}')

Server URL: https://fidelity-scoured-moonbeam.ngrok-free.dev


In [5]:
# Client config
CLIENT_ID = 'client_2'
DATA_DIR = os.path.join(BASE_DIR, 'data/client_2')

# Training config (sama dengan repo ori)
SEED = 42
N_CLASSES = 2
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 5e-4
LOCAL_EPOCHS = 1
MAX_ROUNDS = 3
TRAIN_RATIO = 0.85

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Client ID: {CLIENT_ID}')
print(f'Data: {DATA_DIR}')
print(f'Server: {SERVER_URL}')
print(f'Device: {device}')

Client ID: client_2
Data: workshop-data/day-3/data/client_2
Server: https://fidelity-scoured-moonbeam.ngrok-free.dev
Device: cpu


## 5. Model Architecture (sama dengan server)

In [6]:
class EfficientNetB0(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.model = EfficientNet.from_pretrained('efficientnet-b0')
        self.num_ftrs = self.model._fc.in_features
        self.model._fc = nn.Linear(self.num_ftrs, n_classes)
        self.projector = nn.Sequential(
            nn.Linear(self.num_ftrs, self.num_ftrs),
            nn.Linear(self.num_ftrs, 1024)
        )

    def forward(self, x, project=False):
        features = self.model.extract_features(x)
        features = self.model._avg_pooling(features)
        features = features.flatten(start_dim=1)
        out = self.model._dropout(features)
        out = self.model._fc(out)
        return features, out

print('Model architecture defined!')

Model architecture defined!


## 6. Dataset & DataLoader

In [7]:
class LocalDataset(Dataset):
    def __init__(self, data_dir, transform, mode='all', seed=42):
        self.transform = transform
        samples = []
        csv_path = os.path.join(data_dir, 'labels.csv')
        images_dir = os.path.join(data_dir, 'images')
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            img_path = os.path.join(images_dir, row['filename'])
            if os.path.exists(img_path):
                samples.append((img_path, int(row['label'])))

        # Train/val split
        if mode in ['train', 'val']:
            np.random.seed(seed)
            indices = np.random.permutation(len(samples))
            split = int(0.85 * len(samples))
            if mode == 'train':
                indices = indices[:split]
            else:
                indices = indices[split:]
            samples = [samples[i] for i in indices]

        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        return self.transform(image), label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create dataloaders
train_dataset = LocalDataset(DATA_DIR, train_transform, mode='train', seed=SEED)
val_dataset = LocalDataset(DATA_DIR, val_transform, mode='val', seed=SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

DATA_SIZE = len(train_dataset)
print(f'Train samples: {DATA_SIZE}')
print(f'Val samples: {len(val_dataset)}')

Train samples: 595
Val samples: 105


## 7. FL Client Functions

In [8]:
def register():
    '''Register client to server'''
    response = requests.post(f'{SERVER_URL}/register', json={
        'client_id': CLIENT_ID,
        'data_size': DATA_SIZE
    })
    result = response.json()
    print(f'Registered! Current round: {result["current_round"]}')
    return result['current_round']

def download_model():
    '''Download global model from server'''
    response = requests.get(f'{SERVER_URL}/model/download', params={'client_id': CLIENT_ID})
    if response.status_code != 200:
        raise Exception('Failed to download model')
    weights = torch.load(io.BytesIO(response.content), map_location=device)
    print('Global model downloaded!')
    return weights

def upload_weights(model, current_round):
    '''Upload weights using chunked upload with retry'''
    print('Uploading weights...')

    # Serialize and compress
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    weights_data = buffer.getvalue()
    compressed = gzip.compress(weights_data, compresslevel=9)
    encoded = base64.b64encode(compressed).decode('utf-8')

    print(f'  Original: {len(weights_data)/1024/1024:.2f} MB')
    print(f'  Compressed: {len(compressed)/1024/1024:.2f} MB')

    # Larger chunk size to reduce requests (2MB instead of 500KB)
    CHUNK_SIZE = 2 * 1024 * 1024
    total_chunks = (len(encoded) + CHUNK_SIZE - 1) // CHUNK_SIZE
    upload_id = hashlib.md5(f'{CLIENT_ID}_{current_round}_{datetime.now()}'.encode()).hexdigest()[:16]

    print(f'  Uploading {total_chunks} chunks...')

    for i in range(total_chunks):
        chunk = encoded[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE]
        # Retry mechanism for unstable connections
        for attempt in range(3):
            try:
                resp = requests.post(f'{SERVER_URL}/model/upload_chunk', json={
                    'client_id': CLIENT_ID,
                    'upload_id': upload_id,
                    'chunk_idx': i,
                    'total_chunks': total_chunks,
                    'chunk_data': chunk,
                    'data_size': DATA_SIZE,
                    'round': current_round
                }, timeout=60)
                if resp.status_code == 200:
                    break
            except Exception as e:
                if attempt < 2:
                    print(f'  Retry chunk {i+1}...')
                    time.sleep(2)
                else:
                    raise e
        print(f'  Chunk {i+1}/{total_chunks} uploaded')

    # Complete upload
    response = requests.post(f'{SERVER_URL}/model/upload_complete', json={
        'client_id': CLIENT_ID,
        'upload_id': upload_id,
        'data_size': DATA_SIZE,
        'round': current_round
    }, timeout=60)
    print('  Upload complete!')
    return response.json()

def wait_for_aggregation(current_round):
    '''Wait for server to complete aggregation'''
    print('Waiting for aggregation...')
    while True:
        try:
            response = requests.get(f'{SERVER_URL}/status', timeout=30)
            status = response.json()
            if status['current_round'] > current_round:
                print(f'Aggregation complete! Round {current_round} -> {status["current_round"]}')
                return status['current_round']
            print(f'  Waiting... ({status["weights_received_count"]}/{status["expected_clients"]} clients)')
        except:
            print('  Connection error, retrying...')
        time.sleep(5)

print('FL client functions defined!')

FL client functions defined!


## 8. Local Training Function

In [9]:
def local_train(model, epochs=LOCAL_EPOCHS):
    '''Perform local training'''
    print(f'Starting local training ({epochs} epochs)...')
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            _, logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            if batch_idx % 10 == 0:
                print(f'  Batch {batch_idx}, Loss: {loss.item():.4f}')

        acc = 100. * correct / total
        avg_loss = running_loss / len(train_loader)
        print(f'Epoch {epoch+1}: Loss={avg_loss:.4f}, Acc={acc:.2f}%')

    print('Local training completed!')
    return model

print('Local training function defined!')

Local training function defined!


## 9. Run FL Client

**PENTING**: Pastikan server.ipynb sudah running!

In [10]:
# Check server connection
try:
    response = requests.get(f'{SERVER_URL}/health', timeout=5)
    print(f'Server connected! Status: {response.json()}')
except:
    print('ERROR: Server not running!')
    print('Jalankan server.ipynb terlebih dahulu!')
    raise Exception('Server not available')

Server connected! Status: {'round': 0, 'status': 'healthy'}


In [11]:
# Register and start FL
current_round = register()
rounds_completed = 0

print('='*50)
print(f'FL CLIENT {CLIENT_ID} STARTING')
print(f'Max rounds: {MAX_ROUNDS}')
print('='*50)

while rounds_completed < MAX_ROUNDS:
    print(f'\n--- Round {current_round} ---')

    # 1. Download global model
    global_weights = download_model()
    model = EfficientNetB0(N_CLASSES).to(device)
    model.load_state_dict(global_weights)

    # 2. Local training
    model = local_train(model)

    # 3. Upload weights
    result = upload_weights(model, current_round)
    print(f'Upload result: {result["status"]}')

    # 4. Wait for aggregation (if not already done)
    if 'aggregation' in result:
        print(f'Aggregation done! BACC: {result["aggregation"]["bacc"]*100:.2f}%')
        current_round = result['new_round']
    else:
        current_round = wait_for_aggregation(current_round)

    rounds_completed += 1
    print(f'Progress: {rounds_completed}/{MAX_ROUNDS} rounds')

print('\n' + '='*50)
print(f'FL CLIENT {CLIENT_ID} COMPLETED!')
print(f'Rounds completed: {rounds_completed}')
print('='*50)

Registered! Current round: 0
FL CLIENT client_2 STARTING
Max rounds: 3

--- Round 0 ---
Global model downloaded!
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 423MB/s]


Loaded pretrained weights for efficientnet-b0
Starting local training (1 epochs)...
  Batch 0, Loss: 0.7601
  Batch 10, Loss: 0.5302
Epoch 1: Loss=0.5893, Acc=68.24%
Local training completed!
Uploading weights...
  Original: 26.83 MB
  Compressed: 24.68 MB
  Uploading 17 chunks...
  Chunk 1/17 uploaded
  Chunk 2/17 uploaded
  Chunk 3/17 uploaded
  Chunk 4/17 uploaded
  Chunk 5/17 uploaded
  Chunk 6/17 uploaded
  Chunk 7/17 uploaded
  Chunk 8/17 uploaded
  Chunk 9/17 uploaded
  Chunk 10/17 uploaded
  Chunk 11/17 uploaded
  Chunk 12/17 uploaded
  Chunk 13/17 uploaded
  Chunk 14/17 uploaded
  Chunk 15/17 uploaded
  Chunk 16/17 uploaded
  Chunk 17/17 uploaded
  Upload complete!
Upload result: received
Aggregation done! BACC: 30.00%
Progress: 1/3 rounds

--- Round 1 ---
Global model downloaded!
Loaded pretrained weights for efficientnet-b0
Starting local training (1 epochs)...
  Batch 0, Loss: 0.5393
  Batch 10, Loss: 0.5184
Epoch 1: Loss=0.4838, Acc=77.31%
Local training completed!
Uploadi